# Quantum Tree Gaussian preparation

Execute the original eight-qubit stage through native qfunc and compile the 127-qubit Circuit extension. Retain the tuned rotations, truncated Fourier schedule and signed probability metric.

## What FlagQuantum supplies

The program below allocates logical bits, applies static product rotations and a controlled helper, then requests pairwise phases. Synthesis checks the borrowed slices, expands the bounded loops and lowers the controlled rotations. There are no physical wire labels in this executable program. The eight-bit result is checked against both the retained Circuit and an independent binary-path reference, including absolute complex phase.

The original 127-bit stage remains an explicit Circuit compilation exercise; it is not simulated and is not claimed as native qfunc execution. Circuit-to-CX/U conversion may introduce a recorded common phase. The Gaussian fit itself is approximate: the source rotation and phase thresholds remain unchanged.

Edit the qfunc below to explore a different logical construction. The explicit `workflow.gaussian_tree` binding makes the stage-one builder use this definition. Classical code supplies the tuned angles; it does not substitute target amplitudes for quantum execution.


Use the local native runtime and the source algorithm parameters.

In [ ]:
from pathlib import Path

from IPython.display import Image, display

import examples.synthesis.community.hackathons.iquhack_2025.winning_gaussian.overview.main as report
import examples.synthesis.community.hackathons.iquhack_2025.winning_gaussian.overview.workflow as workflow

import math

from flagquantum.synth import (
    R,
    SWAP,
    H,
    X,
    Output,
    QArray,
    QBit,
    allocate,
    control,
    phase,
    qfunc,
)


@qfunc
def rotate_y(angle: float, q: QBit) -> None:
    R(angle, math.pi / 2, q)


@qfunc
def gaussian_tree(
    width: int,
    rotations: list[float],
    phases: list[float],
    tweak: float,
    q: Output[QArray[QBit]],
) -> None:
    allocate(width, q)
    for bit in range(width):
        if rotations[bit] > 0.000001:
            R(rotations[bit], math.pi / 2, q[bit])
    control(q[0], lambda: rotate_y(tweak, q[1:2]))
    for bit in range(width):
        if 2 * bit < width - 1:
            SWAP(q[bit], q[width - 1 - bit])
    for bit in range(width):
        H(q[bit])
        for distance in range(1, 9):
            if bit + distance < width:
                if phases[distance - 1] > 0.01:
                    phase(q[bit + distance] * q[bit], phases[distance - 1])
    X(q[0])


workflow.gaussian_tree = gaussian_tree


Run on the local CPU and seed the single stage-one draw.

In [ ]:
figures = Path("notebook-figures")
programs = Path("notebook-programs")


Use native qfunc for the executable stage, Circuit for the large compile-only stage, and Matplotlib for the result.


In [ ]:
assert workflow.BASIS == ("cx", "u")


Retain the tuned product rotations and truncated Fourier schedule.

In [ ]:
assert (workflow.ROTATION_FLOOR, workflow.PHASE_FLOOR) == (1e-6, 1e-2)


Keep the original stage width ceilings of 18 and 127.

In [ ]:
assert workflow.SIMULATED_WIDTH == 8 and workflow.SYNTHESIS_WIDTH == 127


Synthesize the original eight-qubit stage with zero work bits, then convert it to the reported CX/U basis.


In [ ]:
stage_1 = workflow.build(8, ceiling=18)


Inspect the first-stage program width.

In [ ]:
assert stage_1.program.n_wires == 8


Simulate the first stage and draw one actual sample.

In [ ]:
observed = workflow.simulate(stage_1)


Retain the complete signed probability grid.

In [ ]:
assert len(observed.probabilities) == 256


Compile the original 127-qubit second stage.

In [ ]:
stage_2 = workflow.build(127, ceiling=127)


Inspect the second-stage program width.

In [ ]:
assert stage_2.program.n_wires == 127


Evaluate the mean squared error on every signed fixed-point coordinate.

In [ ]:
report.report_simulation(observed)


Use the native static resource estimator.

In [ ]:
assert stage_1.resources.basis == "static_instruction_sequence"


Compare actual and theoretical distributions and show first-stage resources.

In [ ]:
report.report(stage_1, 1)
report.plot(observed, figures)
display(Image(filename=str(figures / "gaussian-comparison.png")))


Report the full-scale second-stage operation counts.

In [ ]:
report.report(stage_2, 2)


Export round-trip checked native circuit IR.

In [ ]:
save_program = workflow.save_program


Save both native stages under the original Quantum Tree name.

In [ ]:
saved = (save_program(stage_1, programs, 1), save_program(stage_2, programs, 2))
print("Saved native program artifacts:", tuple(path.name for path in saved))
